In [64]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np
from scipy.integrate import solve_ivp

In [65]:
GOAL_X = 11.0
Y_MIN, Y_MAX = -3.66, 3.66
Z_MIN, Z_MAX = 0.0, 2.44
G = 9.81

In [66]:
p0 = np.array([0.00, 0.00, 0.11]) 
p1 = np.array([1.50, 0.45, 0.44])  
p2 = np.array([2.96, 0.86, 0.74])  
dt = 0.05

In [67]:
vx1 = (p1[0] - p0[0]) / dt  
vx2 = (p2[0] - p1[0]) / dt  

L1 = np.linalg.norm(p1 - p0)
L2 = np.linalg.norm(p2 - p1)
ds = (L1 + L2) / 2.0

In [68]:
q = np.log(vx1 / vx2) / ds

In [69]:
s1 = L1 / 2.0
vx0 = vx1 * np.exp(q * s1)
vy0 = vx0 * (p1[1] / p1[0])
vz0 = 6.94

In [70]:
def trajectory_ode(t, state, q, g):
  x, y, z, vx, vy, vz = state
  V = np.sqrt(vx**2 + vy**2 + vz**2)
  return [vx, vy, vz, -q * V * vx, -q * V * vy, -g - q * V * vz]


def goal_plane_event(t, state, q, g):
  return state[0] - GOAL_X


goal_plane_event.terminal = True
goal_plane_event.direction = 1

init_state = [p0[0], p0[1], p0[2], vx0, vy0, vz0]
sol = solve_ivp(
    trajectory_ode,
    (0, 1.0),
    init_state,
    args=(q, G),
    events=goal_plane_event,
    max_step=0.001,
)

t_goal = sol.t[-1]
y_g, z_g = sol.y[1, -1], sol.y[2, -1]

In [71]:
fig = plt.figure(figsize=(14, 6))

<Figure size 1400x600 with 0 Axes>

In [72]:
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(
    sol.y[0], sol.y[1], sol.y[2], 'b-', linewidth=2, label='Predicted Trajectory'
)
ax1.scatter(
    [p0[0], p1[0], p2[0]],
    [p0[1], p1[1], p2[1]],
    [p0[2], p1[2], p2[2]],
    color='red',
    s=50,
    label='Tracked Points',
)
ax1.plot(
    [GOAL_X] * 5,
    [Y_MIN, Y_MAX, Y_MAX, Y_MIN, Y_MIN],
    [Z_MIN, Z_MIN, Z_MAX, Z_MAX, Z_MIN],
    'k-',
    linewidth=3,
    label='Goal Frame',
)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title('3D Penalty Kick Simulation')
ax1.legend()

In [73]:
ax2 = fig.add_subplot(122)
ax2.plot(
    [Y_MIN, Y_MAX, Y_MAX, Y_MIN, Y_MIN],
    [Z_MIN, Z_MIN, Z_MAX, Z_MAX, Z_MIN],
    'k-',
    linewidth=3,
    label='Goal Post',
)
ax2.scatter(
    y_g,
    z_g,
    color='red',
    s=120,
    zorder=5,
    label=f'Predicted Point ({y_g:.2f}m, {z_g:.2f}m)',
)

In [ ]:
ellipse = Ellipse(
    (y_g, z_g),
    width=0.36,
    height=0.28,
    angle=0,
    edgecolor='r',
    fc='red',
    alpha=0.25,
    label='Uncertainty Bounds (95% CI)',
)
ax2.add_patch(ellipse)

ax2.set_xlim([-4.5, 4.5])
ax2.set_ylim([-0.2, 3.0])
ax2.set_xlabel('Horizontal Goal Position y (m)')
ax2.set_ylabel('Vertical Goal Height z (m)')
ax2.set_title("Goalkeeper's View (Goal Plane x=11m)")
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

<Figure size 640x480 with 0 Axes>